# micro-datalog Benchmarks

Comparing evaluation strategies: Semi-naive (full materialization), MST (Magic Sets), and SDT (Subsumptive Demand Transformation).

All benchmarks use the Free Join evaluation backend.

Based on Tekle & Liu 2011 and Wang et al. 2024.

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from pymycrodatalog import MicroRuntime, Variable

X = Variable("X")
Y = Variable("Y")
Z = Variable("Z")
U = Variable("U")
V = Variable("V")

## Helpers

In [ ]:
def chain_edges(n: int) -> list[tuple[str, tuple[int, int]]]:
    return [("e", (i, i + 1)) for i in range(n - 1)]


def cycle_edges(n: int) -> list[tuple[str, tuple[int, int]]]:
    return [("e", (i, (i + 1) % n)) for i in range(n)]


def dense_edges(n: int, count: int) -> list[tuple[str, tuple[int, int]]]:
    seen: set[tuple[int, int]] = set()
    result = []
    for i in range(count * 3):
        s, d = (i * 13 + 7) % n, (i * 31 + 11) % n
        if s != d and (s, d) not in seen:
            seen.add((s, d))
            result.append(("e", (s, d)))
        if len(result) >= count:
            break
    return result


def imm_chain(n: int) -> list[tuple[str, tuple[int, int]]]:
    return [("imm", (i, i + 1)) for i in range(n - 1)]


def bench_semi_naive(
    rules: list[tuple],
    facts: list[tuple],
    pred: str,
    pattern: tuple,
    iterations: int = 10,
) -> tuple[float, int]:
    times = []
    count = 0
    for _ in range(iterations):
        rt = MicroRuntime(rules, engine="free_join")
        for f in facts:
            rt.insert(f)
        t0 = time.perf_counter_ns()
        rt.poll()
        results = rt.query(pred, pattern)
        elapsed = time.perf_counter_ns() - t0
        count = len(results)
        times.append(elapsed / 1000)
    times.sort()
    return times[len(times) // 2], count


def bench_strategy(
    rules: list[tuple],
    facts: list[tuple],
    pred: str,
    pattern: tuple,
    strategy: str,
    iterations: int = 10,
) -> tuple[float, int]:
    times = []
    count = 0
    for _ in range(iterations):
        rt = MicroRuntime(rules, engine="free_join")
        for f in facts:
            rt.insert(f)
        t0 = time.perf_counter_ns()
        results = rt.query_program(pred, pattern, rules, strategy)
        elapsed = time.perf_counter_ns() - t0
        count = len(results)
        times.append(elapsed / 1000)
    times.sort()
    return times[len(times) // 2], count


def run_benchmark(
    label: str,
    rules: list[tuple],
    facts: list[tuple],
    pred: str,
    pattern: tuple,
    iterations: int = 10,
) -> pd.DataFrame:
    rows = []
    t, c = bench_semi_naive(rules, facts, pred, pattern, iterations)
    rows.append({"strategy": "Semi-naive", "time_us": t, "results": c})
    for strategy, name in [("Bottom-up", "MST"), ("SDT", "SDT")]:
        t, c = bench_strategy(rules, facts, pred, pattern, strategy, iterations)
        rows.append({"strategy": name, "time_us": t, "results": c})

    df = pd.DataFrame(rows)
    counts = df["results"].unique()
    assert len(counts) == 1, f"{label}: result count mismatch {counts}"
    df.attrs["label"] = label
    df.attrs["result_count"] = int(counts[0])
    return df

## Programs

In [ ]:
LINEAR_TC = [
    (("tc", (X, Y)), ("e", (X, Y))),
    (("tc", (X, Z)), ("e", (X, Y)), ("tc", (Y, Z))),
]

# Paper's running example: rel(x,y) :- imm(x,y). rel(x,y) :- imm(u,v), rel(u,x), rel(v,y).
PAPER_REL = [
    (("rel", (X, Y)), ("imm", (X, Y))),
    (("rel", (X, Y)), ("imm", (U, V)), ("rel", (U, X)), ("rel", (V, Y))),
]

ANCESTOR = [
    (("ancestor", (X, Y)), ("parent", (X, Y))),
    (("ancestor", (X, Z)), ("parent", (X, Y)), ("ancestor", (Y, Z))),
]

## Benchmark 1: Linear TC — BF query

In [ ]:
tc_bf_results = []
for label, facts in [
    ("chain(50)", chain_edges(50)),
    ("chain(100)", chain_edges(100)),
    ("cycle(30)", cycle_edges(30)),
    ("cycle(50)", cycle_edges(50)),
    ("dense(30,100)", dense_edges(30, 100)),
]:
    df = run_benchmark(f"TC BF {label}", LINEAR_TC, facts, "tc", (0, None))
    tc_bf_results.append(df)
    print(f"\n{df.attrs['label']} ({df.attrs['result_count']} results):")
    print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, len(tc_bf_results), figsize=(4 * len(tc_bf_results), 5), sharey=False)
for ax, df in zip(axes, tc_bf_results):
    df.plot.bar(x="strategy", y="time_us", ax=ax, legend=False, color="steelblue")
    ax.set_title(df.attrs["label"], fontsize=10)
    ax.set_ylabel("Median time (\u00b5s)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Linear TC \u2014 BF Query", fontsize=14)
fig.tight_layout()
plt.show()

## Benchmark 2: Linear TC — FF query (full materialization)

In [ ]:
tc_ff_results = []
for label, facts in [
    ("chain(50)", chain_edges(50)),
    ("cycle(20)", cycle_edges(20)),
]:
    df = run_benchmark(f"TC FF {label}", LINEAR_TC, facts, "tc", (None, None), iterations=5)
    tc_ff_results.append(df)
    print(f"\n{df.attrs['label']} ({df.attrs['result_count']} results):")
    print(df.to_string(index=False))

## Benchmark 3: Paper's running example (rel with imm)

In [ ]:
paper_results = []
for label, facts in [
    ("imm(20)", imm_chain(20)),
    ("imm(40)", imm_chain(40)),
    ("imm(60)", imm_chain(60)),
]:
    df = run_benchmark(f"Paper BF {label}", PAPER_REL, facts, "rel", (0, None))
    paper_results.append(df)
    print(f"\n{df.attrs['label']} ({df.attrs['result_count']} results):")
    print(df.to_string(index=False))

In [ ]:
sizes = [20, 40, 60]
fig, ax = plt.subplots(figsize=(10, 6))
for strat in ["Semi-naive", "MST", "SDT"]:
    times = [df[df["strategy"] == strat]["time_us"].values[0] for df in paper_results]
    ax.plot(sizes, times, marker="o", label=strat)
ax.set_xlabel("imm chain size")
ax.set_ylabel("Median time (\u00b5s)")
ax.set_title("Paper Running Example \u2014 Scaling")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## Benchmark 4: Ancestor BB \u2014 multi-hop point query

In [ ]:
ancestor_results = []
for n in [20, 50, 100]:
    facts = [("parent", (i, i + 1)) for i in range(n - 1)]
    df = run_benchmark(f"Ancestor BB chain({n})", ANCESTOR, facts, "ancestor", (0, n - 1))
    ancestor_results.append(df)
    print(f"\n{df.attrs['label']} ({df.attrs['result_count']} results):")
    print(df.to_string(index=False))

## Benchmark 5: Linear TC \u2014 FB query (backward reachability)

In [ ]:
tc_fb_results = []
for label, facts in [
    ("cycle(30)", cycle_edges(30)),
    ("cycle(50)", cycle_edges(50)),
]:
    df = run_benchmark(f"TC FB {label}", LINEAR_TC, facts, "tc", (None, 0))
    tc_fb_results.append(df)
    print(f"\n{df.attrs['label']} ({df.attrs['result_count']} results):")
    print(df.to_string(index=False))

## Summary

In [ ]:
all_dfs = tc_bf_results + tc_ff_results + paper_results + ancestor_results + tc_fb_results
summary_rows = []
for df in all_dfs:
    row = {"benchmark": df.attrs["label"], "results": df.attrs["result_count"]}
    for _, r in df.iterrows():
        row[r["strategy"]] = f"{r['time_us']:.0f}"
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))